# Importando pandas e dando nomes as colunas

In [1]:
import sqlglot
import pandas as pd
from tkinter import ttk
from sqlglot import exp
import customtkinter as ctk

df = pd.read_csv("empregados.txt", sep=";", header=None)
df.columns = ['nome', 'cpf', 'matricula', 'sexo', 'salario', 'idade']
df['nome'] = df['nome'].str.encode('latin1').str.decode('utf-8')

ModuleNotFoundError: No module named 'sqlglot'

# Funções

In [ ]:
def avaliar_condicao(cond, df):

    if isinstance(cond, exp.And):
        return avaliar_condicao(cond.left, df) & avaliar_condicao(cond.right, df)

    if isinstance(cond, exp.Or):
        return avaliar_condicao(cond.left, df) | avaliar_condicao(cond.right, df)

    coluna_esq = cond.left.name
    right = cond.right

    if isinstance(right, exp.Column):
        coluna_dir = right.name

        if isinstance(cond, exp.GT):
            return df[coluna_esq] > df[coluna_dir]
        elif isinstance(cond, exp.LT):
            return df[coluna_esq] < df[coluna_dir]
        elif isinstance(cond, exp.EQ):
            return df[coluna_esq] == df[coluna_dir]

    else:
        valor = right.this

        try:
            valor = float(valor)
        except:
            pass

        if isinstance(cond, exp.GT):
            return df[coluna_esq] > valor
        elif isinstance(cond, exp.LT):
            return df[coluna_esq] < valor

    return None


def select(parsed, df):

    df_filtrado = df

    where = parsed.find(exp.Where)
    if where:
        mascara = avaliar_condicao(where.this, df)
        df_filtrado = df[mascara]

    group = parsed.find(exp.Group)
    if group:
        group_cols = [g.name for g in group.expressions]
        agregacoes = {}

        for e in parsed.expressions:
            if isinstance(e, exp.Avg):
                agregacoes[e.this.name] = 'mean'
            elif isinstance(e, exp.Sum):
                agregacoes[e.this.name] = 'sum'
            elif isinstance(e, exp.Count):
                agregacoes[e.this.name] = 'count'

        return df_filtrado.groupby(group_cols).agg(agregacoes).reset_index()

    order = parsed.find(exp.Order)
    if order:
        col_ordem = order.expressions[0].this.name
        desc = order.expressions[0].args.get("desc")

        df_filtrado = df_filtrado.sort_values(
            by=col_ordem,
            ascending=not desc
        )

    colunas = []

    for c in parsed.expressions:
        if isinstance(c, exp.Star):
            colunas = df.columns.tolist()
            break
        elif isinstance(c, exp.Column):
            colunas.append(c.name)

    return df_filtrado[colunas]


def insert(parsed, df):
    if isinstance(parsed, exp.Insert):
        values = parsed.expression
        linha = values.expressions[0]
        nova_linha = [v.this for v in linha.expressions]
        df.loc[len(df)] = nova_linha

    return df


def mostrar_tabela(df_resultado):

    for item in tree.get_children():
        tree.delete(item)

    tree["columns"] = list(df_resultado.columns)
    tree["show"] = "headings"

    for col in df_resultado.columns:
        tree.heading(col, text=col)
        tree.column(col, anchor="center", width=120)

    for _, row in df_resultado.iterrows():
        tree.insert("", "end", values=list(row))


def mostrar_resultado(res):

    if isinstance(res, pd.DataFrame):
        mostrar_tabela(res)
    else:
        for item in tree.get_children():
            tree.delete(item)

        tree["columns"] = ["Resultado"]
        tree["show"] = "headings"
        tree.heading("Resultado", text="Resultado")

        tree.insert("", "end", values=[str(res)])


def get_query():
    global df

    query = campo.get()

    try:
        parsed = sqlglot.parse_one(query)

        if isinstance(parsed, exp.Select):
            resultado = select(parsed, df)
            mostrar_resultado(resultado)

        elif isinstance(parsed, exp.Insert):
            df = insert(parsed, df)
            mostrar_resultado("Inserido com sucesso!")

    except Exception as e:
        mostrar_resultado(f"Erro: {e}")


# Interface Gráfica

In [ ]:
ctk.set_appearance_mode('dark')

app = ctk.CTk()
app.title('Lista01_SGBD')
app.geometry("1366x768")

campo = ctk.CTkEntry(app, placeholder_text='Digite uma consulta SQL', width=400, height=50)
campo.place(x=10, y=10)

button = ctk.CTkButton(app, text='Enviar', command=get_query, width=400)
button.place(x=10, y=70)

tree = ttk.Treeview(app)
tree.place(x=450, y=10, width=850, height=700)

app.mainloop()